## Create Metric Views for your Genie Agents

### Set the Execution Context

In [ ]:
%sql
USE SCHEMA genie_lab;

### Create the Sales Analysis Metric View

In [ ]:
%sql
CREATE OR REPLACE VIEW sales_analysis AS

SELECT
    o.order_id,
    o.order_date,
    o.quantity,
    o.gross_revenue,
    o.discount_amount,
    o.gross_revenue - o.discount_amount AS net_revenue,
    o.returned,

    c.customer_id,
    c.customer_name,
    c.customer_segment,
    c.region,

    p.product_id,
    p.product_name,
    p.product_category

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

JOIN products p
    ON o.product_id = p.product_id

WHERE o.order_status = 'COMPLETED';

### Add Context to the Metric View 

In [ ]:
%sql
CREATE OR REPLACE METRIC VIEW sales_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1

source: db_context_engineer_workspace.genie_lab.sales_analysis

comment: >
  Governed sales metrics for Contoso.
  Cancelled orders are excluded from this dataset.
  Use this metric view for revenue, order, quantity,
  discount and return analysis.

dimensions:

  - name: Order Date
    expr: order_date

  - name: Region
    expr: region

  - name: Customer Segment
    expr: customer_segment

  - name: Product Category
    expr: product_category

  - name: Product Name
    expr: product_name

measures:

  - name: Net Revenue
    expr: SUM(net_revenue)

  - name: Gross Revenue
    expr: SUM(gross_revenue)

  - name: Total Discounts
    expr: SUM(discount_amount)

  - name: Total Orders
    expr: COUNT(DISTINCT order_id)

  - name: Units Sold
    expr: SUM(quantity)

  - name: Returned Orders
    expr: SUM(CASE WHEN returned THEN 1 ELSE 0 END)

  - name: Average Order Value
    expr: SUM(net_revenue) / COUNT(DISTINCT order_id)

  - name: Return Rate
    expr: >
      SUM(CASE WHEN returned THEN 1 ELSE 0 END)
      / COUNT(DISTINCT order_id)
$$;

### Query the Metric View

In [ ]:
SELECT
    region,
    MEASURE(`Net Revenue`) AS net_revenue,
    MEASURE(`Total Orders`) AS total_orders,
    MEASURE(`Average Order Value`) AS average_order_value

FROM sales_metrics

GROUP BY region

ORDER BY net_revenue DESC;

In [ ]:
SELECT
    product_category,
    MEASURE(`Net Revenue`) AS net_revenue,
    MEASURE(`Return Rate`) AS return_rate

FROM sales_metrics

GROUP BY product_category;